# Module 9 · 音訊 4：案例 — 環境聲音分類（現代管線）

> **本節定位｜整合案例**
> 把音訊 `01–03`（波形、log-mel、預訓練嵌入）整合到**聲音事件分類**。
> 示範 2026 常見作法：**預訓練音訊模型抽嵌入 + 傳統分類器**。
> CPU 可跑（少量樣本），並含**離線後備合成音訊**。

## 學習目標
- 建立音訊分類流程：載入 → 標準化(16k/mono) → 抽嵌入 → 分類 → 評估。
- 對比「經典 MFCC 特徵」與「預訓練音訊嵌入」。
- 理解「想再更好 → 端到端微調」是下一步（M11）。

In [ ]:
import numpy as np

# 準備少量音訊：優先用 HF 上的小型語音資料；否則合成兩類可分的訊號
def load_audio_samples(n_per_class=6, sr=16000):
    rs = np.random.RandomState(0)
    # 類別 0：低頻 + 少量雜訊；類別 1：高頻 + 較多雜訊（刻意可分，僅示意流程）
    def gen(freq, noise):
        t = np.linspace(0, 1.0, sr, endpoint=False)
        return (np.sin(2*np.pi*freq*t) + noise*rs.randn(sr)).astype(np.float32)
    c0 = [gen(220, 0.05) for _ in range(n_per_class)]
    c1 = [gen(1500, 0.20) for _ in range(n_per_class)]
    waves = c0 + c1
    labels = np.array([0]*n_per_class + [1]*n_per_class)
    return waves, labels, sr, "合成音訊（兩類可分，示意流程；真實任務請用 UrbanSound8K 等）"

waves, labels, sr, source = load_audio_samples()
print(f"資料來源：{source}")
print(f"樣本數 = {len(waves)}（每段 1 秒、16kHz）")

## 路線 A（經典）：MFCC 聚合特徵 + 邏輯回歸

In [ ]:
try:
    import librosa
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    def mfcc_feat(w):
        m = librosa.feature.mfcc(y=w, sr=sr, n_mfcc=13)
        return np.concatenate([m.mean(axis=1), m.std(axis=1)])   # (26,)
    X_mfcc = np.stack([mfcc_feat(w) for w in waves])
    print(f"MFCC 特徵矩陣: {X_mfcc.shape}")
    acc = cross_val_score(LogisticRegression(max_iter=1000), X_mfcc, labels, cv=3).mean()
    print(f"[MFCC + LogReg] 交叉驗證 Accuracy = {acc:.3f}")
except Exception as e:
    print("（未安裝 librosa，略過路線 A）", e)

## 路線 B（現代）：預訓練音訊嵌入 + 邏輯回歸

In [ ]:
try:
    import torch
    from transformers import AutoFeatureExtractor, AutoModel
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import cross_val_score

    fe = AutoFeatureExtractor.from_pretrained("facebook/wav2vec2-base-960h")
    model = AutoModel.from_pretrained("facebook/wav2vec2-base-960h")
    model.eval()

    embs = []
    with torch.no_grad():
        for w in waves:
            inp = fe(w, sampling_rate=sr, return_tensors="pt")
            emb = model(**inp).last_hidden_state.mean(dim=1).squeeze(0)   # (D,)
            embs.append(emb.numpy())
    X_emb = np.stack(embs)
    print(f"音訊嵌入矩陣: {X_emb.shape}  (N, D)")
    acc = cross_val_score(LogisticRegression(max_iter=1000), X_emb, labels, cv=3).mean()
    print(f"[wav2vec2 嵌入 + LogReg] 交叉驗證 Accuracy = {acc:.3f}")
except Exception as e:
    print("（未能下載 wav2vec2，略過路線 B）", e)

## 小結與延伸

| 路線 | 特徵 | 特性 |
|:--|:--|:--|
| A. MFCC + LogReg | 手工聚合特徵 | 輕量、可解釋，無語意 |
| B. wav2vec2 嵌入 + LogReg | 預訓練表示 | 帶語意、較穩健、可遷移 |

**想要最高準確率？** 兩條都是「凍結特徵」。把音訊模型（如 wav2vec2）
**端到端微調**做分類、或微調 **Whisper** 做語音辨識，是 **Module 11 · `04_audio_downstream`** 的內容。